# External KG Experiment (Isolated)

이 노트북은 **기존 A/B/C/D 결과를 수정하지 않고** 외부 리서치 기반 E 조건만 별도로 실행합니다.

In [ ]:
import subprocess
from pathlib import Path
import pandas as pd

BASE = Path('/Users/yiji/Desktop/work/pseudo_lab/game_translation_exp')
SCRIPTS = BASE / 'scripts'
DATA = BASE / 'data'
OUT = BASE / 'outputs'
EVAL = BASE / 'eval'
RUN_DATE = '2026-04-12'


In [ ]:
# 1) Build external relation context
cmd = [
    'python3', str(SCRIPTS / 'build_relation_context_external.py'),
    '--samples-csv', str(DATA / 'samples.csv'),
    '--edges-csv', str(DATA / 'relation_kg_external' / 'relation_edges_external_v1.csv'),
    '--out-csv', str(DATA / 'relation_kg_external' / 'sample_relation_context_external.csv')
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

ctx = pd.read_csv(DATA / 'relation_kg_external' / 'sample_relation_context_external.csv')
print('rows:', len(ctx))
ctx.head(10)


In [ ]:
# 2) Run isolated external condition E
DRY_RUN = True
cmd = [
    'python3', str(SCRIPTS / 'run_condition_e_external.py'),
    '--run-date', RUN_DATE,
]
if DRY_RUN:
    cmd.append('--dry-run')
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True)

e_path = OUT / f'run_{RUN_DATE}_external' / 'E_outputs_external.csv'
e = pd.read_csv(e_path)
print('E rows:', len(e))
e.head(10)


In [ ]:
# 3) Create isolated eval sheet
samples = pd.read_csv(DATA / 'samples.csv')
e = pd.read_csv(OUT / f'run_{RUN_DATE}_external' / 'E_outputs_external.csv')

m = samples[['sample_id','sentence_type','source_text']].copy()
m = m.merge(e[['sample_id','translation']].rename(columns={'translation':'condition_E_external'}), on='sample_id', how='left')

for c in ['E_meaning','E_term','E_consistency','E_naturalness','E_ui_fit','best','error_type','notes']:
    m[c] = ''

out_eval = EVAL / f'eval_sheet_prefilled_E_external_{RUN_DATE}.csv'
m.to_csv(out_eval, index=False, encoding='utf-8')
print('saved:', out_eval)
print('rows:', len(m))
